In [1]:
"""Setup: find competition data and add to path."""
import sys, os, glob
from pathlib import Path

sys.argv = [sys.argv[0]]

for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    root = str(Path(candidate).parent)
    if root not in sys.path:
        sys.path.insert(0, root)
    print(f'Dataset root: {root}')
    break

print('Setup complete')

Dataset root: /kaggle/input/competitions/ai-agent-security-multi-step-tool-attacks
Setup complete


In [2]:
"""Write attack.py and submission.csv."""
import base64
from pathlib import Path

data = "..."
(Path("/kaggle/working") / "attack.py").write_text(base64.b64decode(data).decode())
(Path("/kaggle/working") / "submission.csv").write_text("Id,Score\ngpt_oss_public,0.0\ngpt_oss_private,0.0\ngemma_public,0.0\ngemma_private,0.0\n")
print(f"attack.py: {(Path('/kaggle/working') / 'attack.py').stat().st_size}B", flush=True)

attack.py: 17818B


In [3]:
"""Start JEDAttack inference server."""
import kaggle_evaluation.jed_attack_134815.jed_attack_inference_server
kaggle_evaluation.jed_attack_134815.jed_attack_inference_server.JEDAttackInferenceServer().serve()